# Telco Customer Churn Prediction

> **Goal:** Build a binary classification model to identify customers likely to churn,
> enabling proactive retention strategies before they leave.

**Dataset:** [Telco Customer Churn — Kaggle](https://www.kaggle.com/blastchar/telco-customer-churn)  
**Best Model:** Gradient Boosting + Threshold Tuning  
**Deployment:** FastAPI + Docker

---
## Table of Contents
1. [Imports & Config](#1)
2. [Load & Inspect Data](#2)
3. [Data Cleaning](#3)
4. [Exploratory Data Analysis](#4)
5. [Preprocessing & Splits](#5)
6. [Model Comparison](#6)
7. [Best Model — Gradient Boosting + Threshold Tuning](#7)
8. [Final Evaluation](#8)
9. [Feature Importance](#9)
10. [Model Export](#10)
11. [Inference Example](#11)

<a id='1'></a>
## 1. Imports & Config

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
DATA_PATH    = '../data/Telco-Customer-Churn-data.csv'
MODELS_DIR   = '../models/'

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

print('All imports loaded successfully.')

<a id='2'></a>
## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head().T

In [ ]:
df.info()

<a id='3'></a>
## 3. Data Cleaning

In [ ]:
# Standardise column names
df.columns = df.columns.str.lower().str.strip()

# TotalCharges has 11 blank-string rows -> convert and fill with MonthlyCharges
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
df['totalcharges'] = df['totalcharges'].fillna(df['monthlycharges'])

# SeniorCitizen is 0/1 int -> cast to str for uniform OHE treatment
df['seniorcitizen'] = df['seniorcitizen'].astype(str)

# Encode target: Yes -> 1, No -> 0
df['churn'] = (df['churn'] == 'Yes').astype(int)

# Drop ID column
df.drop(columns=['customerid'], inplace=True)

print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Churn rate     : {df["churn"].mean():.2%}  <- class imbalance!')

<a id='4'></a>
## 4. Exploratory Data Analysis

In [ ]:
# Churn distribution
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['churn'].value_counts()
bars = ax.bar(['No Churn', 'Churn'], counts.values, color=['steelblue', 'salmon'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}\n({val/len(df):.1%})', ha='center', va='bottom', fontsize=10)
ax.set_title('Churn Distribution', fontsize=13)
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# Define feature groups (used throughout notebook)
numerical   = ['tenure', 'monthlycharges', 'totalcharges']
categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice',
    'onlinesecurity', 'onlinebackup', 'deviceprotection',
    'techsupport', 'streamingtv', 'streamingmovies',
    'contract', 'paperlessbilling', 'paymentmethod'
]

# Numerical features by churn
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numerical):
    for label, color, name in zip([0, 1], ['steelblue', 'salmon'], ['No Churn', 'Churn']):
        df[df['churn'] == label][col].plot.hist(ax=ax, alpha=0.6, bins=30, color=color, label=name)
    ax.set_title(col.title())
    ax.legend()
plt.suptitle('Numerical Features by Churn', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by key categorical features
key_cats = ['contract', 'internetservice', 'paymentmethod', 'paperlessbilling']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flatten(), key_cats):
    churn_rate = df.groupby(col)['churn'].mean().sort_values(ascending=False)
    bars = ax.bar(churn_rate.index, churn_rate.values, color='salmon', edgecolor='white')
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.0%}', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'Churn Rate by {col.title()}')
    ax.set_ylabel('Churn Rate')
    ax.set_xticklabels(churn_rate.index, rotation=20, ha='right')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(df[numerical + ['churn']].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. Preprocessing & Splits

Split strategy: **60% train / 20% validation / 20% test** — all stratified on churn.

In [ ]:
X = df.drop(columns=['churn'])
y = df['churn']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Train : {len(X_train):>5} rows  |  churn rate: {y_train.mean():.2%}')
print(f'Val   : {len(X_val):>5} rows  |  churn rate: {y_val.mean():.2%}')
print(f'Test  : {len(X_test):>5} rows  |  churn rate: {y_test.mean():.2%}')

In [ ]:
# Shared preprocessor reused across all models
preprocessor = ColumnTransformer([
    ('ohe',    OneHotEncoder(drop='first', handle_unknown='ignore'), categorical),
    ('scaler', StandardScaler(),                                      numerical),
])

def make_pipeline(model):
    return Pipeline([('preprocessor', preprocessor), ('model', model)])

<a id='6'></a>
## 6. Model Comparison

Key metric: **F1-Score** and **ROC-AUC** — both handle class imbalance better than plain accuracy.

In [ ]:
def evaluate(pipe, X, y):
    y_pred  = pipe.predict(X)
    y_proba = pipe.predict_proba(X)[:, 1]
    return {
        'Accuracy':  round(accuracy_score(y, y_pred), 4),
        'Precision': round(precision_score(y, y_pred), 4),
        'Recall':    round(recall_score(y, y_pred), 4),
        'F1':        round(f1_score(y, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y, y_proba), 4),
    }

candidates = {
    'LR — Baseline':    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LR — Balanced':    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest':    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                                     max_depth=4, subsample=0.8, random_state=RANDOM_STATE),
}

results, trained_pipes = {}, {}

for name, model in candidates.items():
    pipe = make_pipeline(model)
    pipe.fit(X_train, y_train)
    results[name]      = evaluate(pipe, X_val, y_val)
    trained_pipes[name] = pipe
    print(f'{name:<25} | F1: {results[name]["F1"]:.4f} | AUC: {results[name]["ROC-AUC"]:.4f}')

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette('Set2', len(candidates))

for ax, metric in zip(axes, ['F1', 'ROC-AUC']):
    vals = results_df[metric].sort_values(ascending=True)
    bars = ax.barh(vals.index, vals.values, color=colors, edgecolor='white')
    ax.set_xlim(vals.min() - 0.05, vals.max() + 0.05)
    for bar, val in zip(bars, vals.values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)
    ax.set_title(f'Validation {metric}', fontsize=12)
    ax.set_xlabel(metric)

plt.suptitle('Model Comparison — Validation Set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

<a id='7'></a>
## 7. Best Model — Gradient Boosting + Threshold Tuning

The default threshold of **0.5** is suboptimal for imbalanced data.  
We search for the threshold that **maximises F1** on the validation set.

In [ ]:
best_pipe = trained_pipes['Gradient Boosting']
val_proba = best_pipe.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.20, 0.65, 0.01)
f1_scores  = [f1_score(y_val, (val_proba >= t).astype(int)) for t in thresholds]

best_threshold = thresholds[np.argmax(f1_scores)]
print(f'Best threshold : {best_threshold:.2f}')
print(f'Best val F1    : {max(f1_scores):.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores, color='darkorange', linewidth=2)
ax.axvline(best_threshold, color='red', linestyle='--', label=f'Best = {best_threshold:.2f}')
ax.axvline(0.5, color='grey', linestyle=':', label='Default = 0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('F1-Score')
ax.set_title('F1-Score vs Decision Threshold (Validation Set)')
ax.legend()
plt.tight_layout()
plt.show()

<a id='8'></a>
## 8. Final Evaluation on Test Set

In [ ]:
test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= best_threshold).astype(int)

print('=' * 45)
print(f'  Accuracy  : {accuracy_score(y_test, test_pred):.4f}')
print(f'  Precision : {precision_score(y_test, test_pred):.4f}')
print(f'  Recall    : {recall_score(y_test, test_pred):.4f}   <- catches more churners')
print(f'  F1-Score  : {f1_score(y_test, test_pred):.4f}')
print(f'  ROC-AUC   : {roc_auc_score(y_test, test_proba):.4f}')
print('=' * 45)
print()
print(classification_report(y_test, test_pred, target_names=['No Churn', 'Churn']))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred, display_labels=['No Churn', 'Churn'], cmap='Blues', ax=ax1
)
ax1.set_title(f'Confusion Matrix (threshold={best_threshold:.2f})')

RocCurveDisplay.from_predictions(y_test, test_proba, ax=ax2, color='darkorange')
ax2.plot([0, 1], [0, 1], linestyle='--', color='grey')
ax2.set_title('ROC Curve — Test Set')

plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Feature Importance

In [ ]:
ohe_features = (
    best_pipe.named_steps['preprocessor']
    .named_transformers_['ohe']
    .get_feature_names_out(categorical)
    .tolist()
)
feature_names = ohe_features + numerical
importances   = best_pipe.named_steps['model'].feature_importances_

imp_df = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1],
        color=sns.color_palette('flare', len(imp_df)), edgecolor='white')
ax.set_title('Top 20 Feature Importances — Gradient Boosting', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

<a id='10'></a>
## 10. Model Export

Saving the pipeline **and** the best threshold together so the API always uses the right cutoff.

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_PATH = MODELS_DIR + 'churn_pipeline.pkl'
artifact   = {'pipeline': best_pipe, 'threshold': best_threshold}

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(artifact, f)

print(f'Saved -> {MODEL_PATH}')
print(f'  Threshold : {best_threshold:.2f}')

<a id='11'></a>
## 11. Inference Example

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    artifact = pickle.load(f)

loaded_pipeline = artifact['pipeline']
threshold       = artifact['threshold']

# High-risk customer: month-to-month contract, fiber optic, no security, short tenure
sample = pd.DataFrame([{
    'gender': 'Female',  'seniorcitizen': '0',  'partner': 'No',  'dependents': 'No',
    'phoneservice': 'Yes', 'multiplelines': 'No', 'internetservice': 'Fiber optic',
    'onlinesecurity': 'No', 'onlinebackup': 'No', 'deviceprotection': 'No',
    'techsupport': 'No', 'streamingtv': 'Yes', 'streamingmovies': 'Yes',
    'contract': 'Month-to-month', 'paperlessbilling': 'Yes',
    'paymentmethod': 'Electronic check',
    'tenure': 5, 'monthlycharges': 90.0, 'totalcharges': 450.0
}])

prob = loaded_pipeline.predict_proba(sample)[0, 1]
pred = int(prob >= threshold)

print(f'Churn Probability : {prob:.2%}')
print(f'Threshold         : {threshold:.2f}')
print(f'Prediction        : {"CHURN" if pred == 1 else "NO CHURN"}')